# YOLOv8-seg: eksperimen cepat & inspeksi

Jalankan dari folder proyek yang berisi `dataset/` dan `segmentation.yolo/`.
Empat kombinasi: `{raw, gray_world} x {mask_based, bbox_based}`.
Urutan: cek COCO -> build -> inspeksi sampel -> (verifikasi GT) -> training -> evaluasi -> trace dengan prediksi.

In [ ]:
import sys
from itertools import product
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd()))  # agar paket `yolo` bisa di-import

from segmentation.yolo.config import MODES, SPLITS, VARIANTS, Config
from segmentation.yolo.logger import get_logger

cfg = Config(root=Path.cwd())  # ubah bila notebook berada di folder lain
log = get_logger(log_file=cfg.logs_dir / "notebook.log")

## 1. Cek COCO
Pastikan `segmentation_types` = `list` (bukan RLE) dan `category_ids` hanya `1`.

In [ ]:
from segmentation.yolo.coco import load_coco, summarize

for split in SPLITS:
    print(split, summarize(load_coco(cfg.annotation_path(split))), "\n")

## 2. Build dataset
Satu kombinasi dulu untuk uji; semua kombinasi di sel berikutnya.

In [ ]:
from segmentation.yolo.dataset_builder import build_dataset

build_dataset(cfg, "raw", "bbox_based", log=log)

In [ ]:
# Semua kombinasi (jalankan setelah hasil inspeksi di bawah terlihat benar)
for variant, mode in product(VARIANTS, MODES):
    build_dataset(cfg, variant, mode, log=log)

## 3. Inspeksi sampel & riwayatnya
`describe` = riwayat teks; `trace` = visualisasi mentah -> varian -> input model (+GT).

In [ ]:
from segmentation.yolo.history import describe, load_manifest, trace

manifest = load_manifest(cfg.dataset_dir("raw", "bbox_based"))
multi = [r for r in manifest.values() if r.n_polygons > 1]
record = multi[0] if multi else next(iter(manifest.values()))

print("\n".join(describe(record)))
trace(cfg, record)

In [ ]:
# Sampel yang memicu fallback atau tanpa polygon (perlu dicek manual)
columns = ["key", "n_polygons", "crop_fallback", "crop_box"]
rows = [
    {"key": r.key, "n_polygons": r.n_polygons, "crop_fallback": r.crop_fallback, "crop_box": r.crop_box}
    for r in manifest.values()
    if r.crop_fallback or r.n_polygons == 0
]
pd.DataFrame(rows, columns=columns)

## 4. Verifikasi GT: polygon JSON vs `rawMask`
Jika U-Net memakai `rawMask/`, pastikan kedua sumber GT setara. IoU per gambar harus mendekati 1.0.

In [ ]:
import cv2

from segmentation.yolo.coco import group_by_image
from segmentation.yolo.evaluation import binary_scores, ground_truth_mask

split = "val"
coco = load_coco(cfg.annotation_path(split))
anns_by_image = group_by_image(coco)

ious = {}
for info in coco["images"]:
    stem = Path(info["file_name"]).stem
    mask = cv2.imread(str(cfg.root / "dataset" / "rawMask" / split / f"{stem}.png"), cv2.IMREAD_GRAYSCALE)
    assert mask is not None, f"rawMask tidak ditemukan untuk {stem}"
    gt = ground_truth_mask(anns_by_image.get(info["id"], []), info["height"], info["width"])
    ious[stem] = binary_scores(gt, (mask > 127).astype("uint8") * 255)["iou"]

ious = pd.Series(ious)
print(ious.describe())
ious.nsmallest(5)

## 5. Training
Uji cepat memakai `name=` terpisah agar tidak menimpa run penuh.

In [ ]:
from segmentation.yolo.training import train

# Uji cepat: 3 epoch (folder run: runs/smoke_raw_mask_based)
build_dataset(cfg, "raw", "mask_based", log=log)
train(cfg, "raw", "mask_based", epochs=3, batch=8, name="smoke_raw_mask_based", log=log)

In [ ]:
# Training penuh semua kombinasi (pastikan semua dataset sudah di-build)
for variant, mode in product(VARIANTS, MODES):
    train(cfg, variant, mode, epochs=100, batch=16, log=log)

## 6. Evaluasi
- `mask_map50*`, `box_map50`: metrik instance Ultralytics (di ruang input model).
- `semantic_iou`, `semantic_dice`: semua instance digabung, dipetakan balik ke ukuran asli, dibandingkan dengan GT penuh. Gunakan kolom ini untuk membandingkan keempat kombinasi (dan dengan U-Net).

In [ ]:
from segmentation.yolo.evaluation import evaluate_semantic
from segmentation.yolo.training import validate

rows = []
for variant, mode in product(VARIANTS, MODES):
    weights = cfg.best_weights(variant, mode)
    if not weights.exists():
        continue  # kombinasi ini belum dilatih
    rows.append({
        "variant": variant, "mode": mode,
        **validate(cfg, weights, variant, mode, "test", log=log),
        **evaluate_semantic(cfg, weights, variant, mode, "test", log=log),
    })
pd.DataFrame(rows)

## 7. Trace dengan prediksi
Dari citra mentah sampai prediksi dipetakan balik ke koordinat citra asli.

In [ ]:
from ultralytics import YOLO

from segmentation.yolo.evaluation import predict_mask

variant, mode = "raw", "bbox_based"
model = YOLO(str(cfg.best_weights(variant, mode)))
manifest = load_manifest(cfg.dataset_dir(variant, mode))

key = next(k for k in manifest if k.startswith("test/"))  # ganti sesuai kebutuhan
record = manifest[key]
print("\n".join(describe(record)))
trace(cfg, record, pred_mask=predict_mask(model, cfg, record), save_to=cfg.root / "trace_demo.png");

## Catatan
- `trace` merekam transformasi geometri deterministik (crop + resize). Augmentasi saat training (mosaic, flip, HSV) tidak dicatat dan tidak relevan untuk inferensi.
- Pada `bbox_based`, crop memakai bbox *ground truth* (oracle), termasuk di test. Skornya batas atas, bukan performa deployment.
- Log tahapan: `logs/*.log`. Kurva dan metrik per epoch dari Ultralytics: `runs/<nama>/results.csv`.